In [1]:
import pandas as pd


url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/titanic.csv"
df = pd.read_csv(url)

print(df['Survived'].value_counts(normalize=True))

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64


- Pregunta A (Sumarización Categórica): Usen la función .value_counts(normalize=True) en la columna Survived (0 = Murió, 1 = Sobrevivió). ¿Cuál es la tasa de supervivencia global del barco expresada en porcentaje? 

El 61.16% murió y el 38.38% sobrevivió.

In [2]:
df.groupby('Sex')["Survived"].mean()

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

- Pregunta B (Agrupación y Agregación): El famoso código marítimo era "mujeres y niños primero". Comprobemos esto matemáticamente. Ejecuten una agrupación por la columna de género y calculen el promedio de la columna Survived. ¿Qué porcentaje exacto de mujeres sobrevivió en contraste con los hombres?

Mujeres un 74.2% y Hombres un 18.89%. Faltan porque algunos valores en el genero pueden ser nulos.

In [13]:
q1 = df['Fare'].quantile(0.25)
q3 = df['Fare'].quantile(0.75)
iqr = q3 - q1

min_outlier = q1 - 1.5*iqr
max_outlier = q3 + 1.5*iqr

df.value_counts('Pclass')[df['Fare'] > max_outlier]

Pclass
1    216
Name: count, dtype: int64

- Pregunta C: El director financiero nota que la tarifa máxima (Fare) cobrada fue de más de 500 libras, mientras que el promedio ronda las 32. Sospecha de outliers. Utilicen las herramientas de dispersión en Pandas para confirmarlo:
    - Calculen el Cuartil 1 (25%) y el Cuartil 3 (75%) de la columna Fare usando el método .quantile([0.25, 0.75]).
    - Calculen matemáticamente el Rango Intercuartílico (IQR = Q3 - Q1).
    - Calculen el límite superior aceptable (Q3 + 1.5 * IQR). Escriban una línea de código para filtrar el DataFrame y descubrir exactamente cuántos pasajeros pagaron una tarifa por encima de ese límite matemático. ¿A qué clase (Pclass) pertenecían la mayoría de ellos?

Fueron 216 boletos que superaron el max_outlier y todos fueron de la pclass 1.

In [15]:
fare_mean = df['Fare'].mean()
fare_median = df['Fare'].median()

print(f"Media del Fare es {fare_mean} y Mediana es {fare_median}")

Media del Fare es 32.204207968574636 y Mediana es 14.4542


- Pregunta D: Calculen la media y la mediana de la columna Fare. Notarán una diferencia enorme entre ambos valores. Matemáticamente, ¿qué significa que la media sea tan superior a la mediana? Si en el futuro utilizamos un algoritmo basado en distancias euclidianas (como K-Nearest Neighbors) sin escalar previamente esta variable, ¿cómo afectará esta asimetría al aprendizaje del modelo?

El que sea tan superior la media a la mediana quiere decir que hay muchos outliers del lado derecho (grandes), que hacen que la media se desplace a valores más grandes. Si no se escalara antes la variable, el modelo puede entender que la mayoría de valores de tarifa son cercanos a la media, pero esto sería un error. 

In [17]:
muestra = df.groupby('Survived', group_keys=False).apply(
    lambda x: x.sample(frac=150 / len(df), random_state=42))

muestra.describe()

C:\Users\Keloc\AppData\Local\Temp\ipykernel_59572\187418046.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  muestra = df.groupby('Survived', group_keys=False).apply(


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,150.000000,150.000000,150.000000,116.000000,150.000000,150.000000,150.000000
mean,429.286667,0.386667,2.280000,30.698276,0.533333,0.460000,34.032722
std,268.329391,0.488618,0.852316,14.099627,1.185294,0.945707,48.720487
min,6.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000
25%,204.500000,0.000000,1.000000,22.000000,0.000000,0.000000,7.895800
50%,377.500000,0.000000,3.000000,29.000000,0.000000,0.000000,14.479150
75%,672.250000,1.000000,3.000000,39.250000,1.000000,1.000000,30.973950
max,886.000000,1.000000,3.000000,70.500000,8.000000,5.000000,262.375000


- Pregunta E: En la semana 2 vimos que el muestreo aleatorio simple es peligroso. El objetivo es entrenar un modelo que prediga la supervivencia (Survived), pero las clases están desbalanceadas. Escriban el código en Pandas para extraer una muestra de exactamente 150 pasajeros garantizando que la proporción de sobrevivientes y no sobrevivientes en la muestra sea idéntica a la de la base de datos completa. ¿Qué sesgo evitan al hacer esto?

Con esto evitamos que obtenga un sesgo hacia los no sobrevivientes, ya que son la mayoría en el dataset.

- Pregunta F: Si ejecutan df.groupby('Survived')['Age'].mean(), Pandas calculará el promedio de edad de los que vivieron y los que murieron. Sin embargo, por defecto, Pandas ignora los valores NaN al calcular la media. Si resulta que la gran mayoría de las edades faltantes (NaN) pertenecían a pasajeros de 3ra clase que murieron, ¿qué sesgo estadístico estamos introduciendo involuntariamente en el resultado de esa función y cómo afectaría la inferencia de nuestro modelo?

Al ignorar involuntariamente las edades de los pasajeros de 3ra clase que murieron, se podría tomar como la gran mayoría de personas de 3ra clase sobrevivieron (que tienen su edad registrada). Si se reemplazara la edad en los valores nulos se podría obtener un resultado más acertado a la realidad y nuestra media sería diferente. 

- Pregunta G: Con el cálculo del IQR en la Pregunta C, determinaron que los boletos de 512 libras son outliers matemáticos. En Machine Learning, los valores atípicos pueden representar errores de captura (ruido que aumenta el error irreducible) o casos especiales válidos (señal). Investigando la naturaleza de un barco de lujo, ¿deberíamos eliminar estas filas con .drop() antes de entrenar nuestro modelo? Justifiquen su respuesta arquitectónica.

No los eliminaría directamente basandome solo en que sean outliers matemáticos, ya que no se puede comprobar si son datos validos que sirvan o no. La naturaleza de un barco de lujo puede hacer que las tarifas cambien con el tiempo o quizas aquellos boletos con altos precios fueron reservados. 

- Pregunta H: Observen la columna Name. Es texto libre (dato no estructurado), pero contiene títulos ocultos como "Mr.", "Mrs.", "Miss." o "Master.". Si lograran extraer ese título usando expresiones regulares en Pandas, podrían hacer un .groupby('Titulo')['Age'].median(). ¿Por qué imputar las edades faltantes basándose en la mediana del "Título" (ej. "Master" = niño, "Mr" = adulto) sería estadísticamente superior y reduciría el error de nuestro futuro modelo, en comparación con usar la mediana global?

Al usar la mediana global solo reemplazamos para no perder datos y mantener la dispersión del dataset, pero al usar la mediana de cada titulo podemos representar de forma más acertada las edades reales de los pasajeros también.

In [18]:
var = df['Survived'].var()
print(var)

0.2367722165474984


Pregunta I: Calculen la varianza de la columna Survived. Dado que es una variable categórica codificada como 0 y 1, el resultado numérico estará cerca de $0.23$. Matemáticamente, ¿qué significaría si la varianza de esta variable fuera exactamente 0.0? ¿Qué pasaría si intentan entrenar un algoritmo de clasificación con un dataset donde la variable de respuesta tiene varianza 0.0?

Si la varianza fuera 0, quiere decir que todos los datos de esta columna son iguales. Si intentaramos entrenar un algoritmo de clasificación con esto solo nos daría el mismo resultado siempre, ya que los datos son iguales. 

In [19]:
df.groupby(['Pclass', 'Sex', 'Embarked'])['PassengerId'].count()

Pclass  Sex     Embarked
1       female  C            43
                Q             1
                S            48
        male    C            42
                Q             1
                S            79
2       female  C             7
                Q             2
                S            67
        male    C            10
                Q             1
                S            97
3       female  C            23
                Q            33
                S            88
        male    C            43
                Q            39
                S           265
Name: PassengerId, dtype: int64

Pregunta J: Ejecuten una agrupación por tres niveles al mismo tiempo y cuenten cuántos pasajeros hay en cada subgrupo: df.groupby(['Pclass', 'Sex', 'Embarked'])['PassengerId'].count(). Notarán que algunos subgrupos tienen 1 o 2 pasajeros. Si un algoritmo intenta extraer reglas de probabilidad de grupos tan pequeños, se enfrentará a la "Maldición de la Dimensionalidad" (Curse of Dimensionality). ¿Qué fenómeno perjudicial (sobreajuste o subajuste) ocurrirá inevitablemente si dejamos que el modelo aprenda reglas basadas en esos grupos de 1 solo pasajero?

Será un sobreajuste porque no tendría suficientes datos para identificar un patrón, entonces va a "memorizar" las características de ese único caso.